# PredictGuard — Phase 1, Stage 6: Grouped Cross-Validation & Hyperparameter Tuning

**Project**: Explainable Predictive Maintenance System  
**Dataset**: Microsoft Azure Predictive Maintenance  
**Stage**: 6 of 6 — Final Stage of Phase 1

---

## Why Grouped CV Over Standard CV?

Standard k-fold splits rows randomly. In predictive maintenance each machine emits thousands of sequential hourly observations — adjacent rows from the same machine are highly correlated. A random split lets the same machine appear in both train and validation, causing **machine leakage**: inflated CV scores that collapse when the model sees a new machine in production.

**`StratifiedGroupKFold(groups=machineID)`** guarantees every row for a given machine appears in exactly one fold, mirroring the true deployment scenario.

## Objective

1. Audit fold structure & confirm zero leakage.
2. Run 5-fold grouped CV on all 3 baseline models.
3. Aggregate mean ± std ± 95% CI per metric.
4. Run `RandomizedSearchCV` on XGBoost (15 iterations).
5. Compare default vs tuned XGBoost on the Final Test set (20 unseen machines).
6. Save `models/best_model.pkl` + all reports & figures.

---
## 0. Imports & Setup

In [ ]:
import logging
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.cross_validation import (
    CrossValidator, FoldAnalyzer, HyperparameterTuner, MetricAggregator,
    _build_xgb, _prepare_xy,
    generate_cv_report,
    plot_cv_metric_distribution, plot_foldwise_metrics,
    plot_before_vs_after_tuning, plot_hyperparam_importance,
    plot_training_time_comparison,
)
from src.train import ModelEvaluator

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)

PROCESSED_DIR = project_root / "data" / "processed"
REPORTS_DIR   = project_root / "reports"
FIGURES_DIR   = REPORTS_DIR / "figures"
MODELS_DIR    = project_root / "models"
N_FOLDS       = 5
RANDOM_SEED   = 42
N_ITER        = 15

---
## 1. Load Data

In [ ]:
dev_df  = pd.read_parquet(PROCESSED_DIR / "development.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test.parquet")

X_dev,  y_dev,  groups_dev, feature_names = _prepare_xy(dev_df)
X_test, y_test, _,          _             = _prepare_xy(test_df)

print(f"Development : {dev_df['machineID'].nunique()} machines | {len(dev_df):,} rows | "
      f"{100*y_dev.mean():.2f}% positive")
print(f"Final Test  : {test_df['machineID'].nunique()} machines | {len(test_df):,} rows | "
      f"{100*y_test.mean():.2f}% positive")
print(f"Features    : {len(feature_names)}")

---
## 2. Fold Analysis & Leakage Audit

In [ ]:
analyzer = FoldAnalyzer(n_folds=N_FOLDS, random_seed=RANDOM_SEED)
fold_stats_df, raw_folds = analyzer.analyze(dev_df)

print(f"Leakage detected in any fold: {fold_stats_df['leakage'].any()}")
fold_stats_df

---
## 3. Grouped 5-Fold Cross-Validation — All 3 Models

> **Note**: Training Random Forest on 700k rows takes ~1.5 min per fold — expect ~8 minutes total for this cell.

In [ ]:
cv = CrossValidator(n_folds=N_FOLDS, random_seed=RANDOM_SEED)
cv_results_df = cv.run(dev_df)
cv_results_df.head(15)

---
## 4. Metric Aggregation: Mean ± Std ± 95% CI

In [ ]:
agg          = MetricAggregator()
summary_df   = agg.aggregate(cv_results_df)
formatted_df = agg.format_table(summary_df)

print("\nCV Summary Table (Primary metric: PR-AUC):")
display(formatted_df)

---
## 5. Hyperparameter Tuning — XGBoost (RandomizedSearchCV)

> 15 iterations × 5-fold CV = 75 XGBoost fits. Expect ~5–10 minutes.

In [ ]:
tuner = HyperparameterTuner(n_folds=N_FOLDS, n_iter=N_ITER, scoring="average_precision", random_seed=RANDOM_SEED)
best_estimator, best_params, search_results_df = tuner.tune(X_dev, y_dev, groups_dev)

best_cv_score = search_results_df["mean_test_score"].iloc[0]
print(f"Best CV PR-AUC: {best_cv_score:.4f}")
print(f"Best Parameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

---
## 6. Default vs Tuned XGBoost — Final Test Set Evaluation

In [ ]:
import time
evaluator = ModelEvaluator()

# Default XGBoost
default_xgb = _build_xgb(y_dev.values, seed=RANDOM_SEED)
t0 = time.time()
default_xgb.fit(X_dev, y_dev)
default_eval = evaluator.evaluate("XGBoost (Default)", default_xgb, X_test, y_test, fit_time_sec=time.time()-t0)

# Tuned XGBoost
tuned_eval = evaluator.evaluate("XGBoost (Tuned)", best_estimator, X_test, y_test, fit_time_sec=0)

comparison = pd.DataFrame([
    {"Model": "Default XGBoost", "PR-AUC": default_eval["pr_auc"], "ROC-AUC": default_eval["roc_auc"],
     "F1": default_eval["f1_score"], "Recall": default_eval["recall"], "Precision": default_eval["precision"]},
    {"Model": "Tuned XGBoost",   "PR-AUC": tuned_eval["pr_auc"],   "ROC-AUC": tuned_eval["roc_auc"],
     "F1": tuned_eval["f1_score"],   "Recall": tuned_eval["recall"],   "Precision": tuned_eval["precision"]},
])
display(comparison)

---
## 7. Generate Figures & Reports

In [ ]:
plot_cv_metric_distribution(cv_results_df, FIGURES_DIR)
plot_foldwise_metrics(cv_results_df, FIGURES_DIR)
plot_before_vs_after_tuning(
    {"pr_auc": default_eval["pr_auc"], "roc_auc": default_eval["roc_auc"],
     "f1_score": default_eval["f1_score"], "recall": default_eval["recall"], "precision": default_eval["precision"]},
    {"pr_auc": tuned_eval["pr_auc"],   "roc_auc": tuned_eval["roc_auc"],
     "f1_score": tuned_eval["f1_score"],  "recall": tuned_eval["recall"],   "precision": tuned_eval["precision"]},
    FIGURES_DIR
)
plot_hyperparam_importance(search_results_df, best_params, FIGURES_DIR)
plot_training_time_comparison(cv_results_df, FIGURES_DIR)

generate_cv_report(
    fold_stats_df=fold_stats_df, cv_summary_df=summary_df, formatted_table=formatted_df,
    best_params=best_params, best_cv_score=float(best_cv_score),
    default_test_metrics={"pr_auc": default_eval["pr_auc"], "roc_auc": default_eval["roc_auc"],
                          "f1_score": default_eval["f1_score"], "recall": default_eval["recall"],
                          "precision": default_eval["precision"]},
    tuned_test_metrics={"pr_auc": tuned_eval["pr_auc"],   "roc_auc": tuned_eval["roc_auc"],
                        "f1_score": tuned_eval["f1_score"],  "recall": tuned_eval["recall"],
                        "precision": tuned_eval["precision"]},
    output_path=REPORTS_DIR / "cross_validation_report.md",
)
print("All figures and reports saved!")

---
## 8. Stage 6 Completion Summary

| Deliverable | Status |
|---|---|
| Fold Statistics & Leakage Audit | ✅ |
| 5-Fold Grouped CV (3 models) | ✅ |
| Mean ± Std ± 95% CI Table | ✅ |
| RandomizedSearchCV (XGBoost) | ✅ |
| Default vs Tuned Comparison | ✅ |
| `models/best_model.pkl` | ✅ |
| 6 Publication-Quality Figures | ✅ |
| `reports/cross_validation_report.md` | ✅ |

### Phase 1 Complete! Ready for Phase 2 (Calibration & Trust)